# Bonus: Unit Commitment

PyPSA supports unit commitment constraints for its components, such as minimum part-load, minimum up time, minimum down time, start up costs, shut down costs and ramp rate restrictions. These constraints are important for accurately modeling the operation of conventional power plants.

This tutorial runs through examples of unit commitment for generators at a single bus.

To enable unit commitment on a component, set its attribute `committable` to `True`.

In [ ]:
import pandas as pd

import pypsa

## Minimum Part Load

**Case description:** In the final snapshot, the load goes below the part-load limit of the coal generator (30%), forcing gas to commit.

In [ ]:
nu = pypsa.Network(snapshots=range(4))

nu.add("Bus", "bus")

nu.add(
    "Generator",
    "coal",
    bus="bus",
    committable=True,
    p_min_pu=0.3,
    marginal_cost=20,
    p_nom=10_000,
)

nu.add(
    "Generator",
    "gas",
    bus="bus",
    committable=True,
    marginal_cost=70,
    p_min_pu=0.1,
    p_nom=1_000,
)

nu.add("Load", "load", bus="bus", p_set=[4_000, 6_000, 5_000, 800])

nu.optimize(log_to_console=False)

In [ ]:
nu.generators_t.status

In [ ]:
nu.generators_t.p

## Minimum Up Time

**Case description:** Gas has a minimum up time, forcing it to be online longer than otherwise necessary, which incurs a standby cost for status up without generation.

In [ ]:
nu = pypsa.Network(snapshots=range(4))

nu.add("Bus", "bus")

nu.add(
    "Generator",
    "coal",
    bus="bus",
    committable=True,
    p_min_pu=0.3,
    marginal_cost=20,
    p_nom=10000,
)

nu.add(
    "Generator",
    "gas",
    bus="bus",
    committable=True,
    stand_by_cost=50,
    marginal_cost=70,
    p_min_pu=0.1,
    up_time_before=0,
    min_up_time=3,
    p_nom=1_000,
)

nu.add("Load", "load", bus="bus", p_set=[4_000, 800, 5_000, 3_000])

nu.optimize(log_to_console=False)

In [ ]:
nu.generators_t.status

In [ ]:
nu.objective

In [ ]:
nu.generators_t.p

## Minimum Down Time

**Case description:** Coal has a minimum down time, forcing it to go off longer than otherwise cost-optimal.

In [ ]:
nu = pypsa.Network(snapshots=range(4))

nu.add("Bus", "bus")

nu.add(
    "Generator",
    "coal",
    bus="bus",
    committable=True,
    p_min_pu=0.3,
    marginal_cost=20,
    min_down_time=2,
    down_time_before=1,
    p_nom=10_000,
)

nu.add(
    "Generator",
    "gas",
    bus="bus",
    committable=True,
    marginal_cost=70,
    p_min_pu=0.1,
    p_nom=4_000,
)

nu.add("Load", "load", bus="bus", p_set=[3_000, 800, 3_000, 8_000])

nu.optimize(log_to_console=False)

In [ ]:
nu.objective

In [ ]:
nu.generators_t.status

In [ ]:
nu.generators_t.p

## Start Up and Shut Down Costs

**Case description:** Now there are costs associated with shut down and start up events, which could incentivise longer up times of generators with high start-up and shut-down costs.

In [ ]:
nu = pypsa.Network(snapshots=range(4))

nu.add("Bus", "bus")

nu.add(
    "Generator",
    "coal",
    bus="bus",
    committable=True,
    p_min_pu=0.3,
    marginal_cost=20,
    min_down_time=2,
    start_up_cost=5_000,
    p_nom=10_000,
)

nu.add(
    "Generator",
    "gas",
    bus="bus",
    committable=True,
    marginal_cost=70,
    p_min_pu=0.1,
    shut_down_cost=25,
    p_nom=4_000,
)

nu.add("Load", "load", bus="bus", p_set=[3_000, 800, 3_000, 8_000])

nu.optimize(log_to_console=False)

In [ ]:
nu.objective

In [ ]:
nu.generators_t.status

In [ ]:
nu.generators_t.p

## Ramp Rate Limits

Ramp rate limits can be set for ramping up and down and are given as percentage of the nominal power that can be ramped up or down per snapshot.

In [ ]:
nu = pypsa.Network(snapshots=range(6))

nu.add("Bus", "bus")

nu.add(
    "Generator",
    "coal",
    bus="bus",
    marginal_cost=20,
    ramp_limit_up=0.1,
    ramp_limit_down=0.2,
    p_nom=10_000,
)

nu.add("Generator", "gas", bus="bus", marginal_cost=70, p_nom=4_000)

nu.add("Load", "load", bus="bus", p_set=[4_000, 7_000, 7_000, 7_000, 7_000, 3_000])

nu.optimize(log_to_console=False)

In [ ]:
nu.generators_t.p